In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df['Serangan_Jantung']

# 1. Binary Classification Decision Trees 

In [3]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [5]:
params = {
    'criterion': ['gini', 'entropy', 'log_loss'], #Kriteria mengukur kualitas. splitGunakan ['squared_error', 'friedman_mse', 'absolute_error'] jika untuk Regressor
    'splitter': ['best', 'random'],  #Strategi yang digunakan untuk memilih split di setiap node
    'max_depth': [None] + list(np.arange(3, 31)), #Mencari dari kedalaman 3 hingga 30, atau None (tanpa batas)
    'min_samples_split': np.arange(2, 21),#Jumlah minimum sampel yang diperlukan untuk memisahkan node internal
    'min_samples_leaf': np.arange(1, 21), #Jumlah minimum sampel yang harus ada di leaf (daun) node
    'max_features': [None, 'sqrt', 'log2'] + list(np.arange(0.1, 1.1, 0.1)), #Jumlah fitur yang dipertimbangkan saat mencari split terbaik
    'ccp_alpha': np.logspace(-4, -1, 10)#Parameter untuk Cost-Complexity Pruning (CCP) untuk mencegah overfitting
}

dct = DecisionTreeClassifier()
random_search  = RandomizedSearchCV(estimator=dct,param_distributions=params,n_iter=20,
                                    cv=6,scoring='accuracy',random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model_dct = random_search.best_estimator_

sfs_dct = SequentialFeatureSelector(estimator=best_model_dct,n_features_to_select=5,direction='forward')
sfs_dct.fit(X_train,y_train)

X_train_selected = sfs_dct.transform(X_train)
X_test_selected = sfs_dct.transform(X_test)

fitur_terpilih = X_train.columns[sfs_dct.get_support()]
best_model_dct.fit(X_train_selected,y_train)

print(f'\nHyperparameter Terbaik: {random_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')


Hyperparameter Terbaik: {'splitter': 'best', 'min_samples_split': np.int64(6), 'min_samples_leaf': np.int64(19), 'max_features': np.float64(0.6), 'max_depth': None, 'criterion': 'gini', 'ccp_alpha': np.float64(0.004641588833612777)}

Akurasi Validasi Terbaik: 74.33

Fitur Terbaik Yang Terpilih:
['Jenis_Kelamin', 'Tipe_Nyeri_Dada', 'Detak_Jantung_Max', 'Angina_Olahraga', 'Oldpeak_ST']


In [9]:
test_accuracy = best_model_dct.score(X_test_selected,y_test)
train_accruacy = best_model_dct.score(X_train_selected,y_train)

y_pred_test = best_model_dct.predict(X_test_selected)
y_prob_test = best_model_dct.predict_proba(X_test_selected)

y_pred_train = best_model_dct.predict(X_train_selected)
y_prob_train = best_model_dct.predict_proba(X_train_selected)

print(f'\nAkurasi Pada Data Test: {test_accuracy*100:.2f}')
print(f'\nAkurasi Pada Data Train: {train_accruacy*100:.2f}')


Akurasi Pada Data Test: 72.67

Akurasi Pada Data Train: 75.50


In [10]:
def evaluate_model(pred,test,prob,evaluate,model_name='Decision Tree'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [11]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 and accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [13]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
           Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss    Status
0  Decision Tree  Training  0.755000   0.766798  0.832618  0.798354  0.809534  0.519533  Good Fit
1  Decision Tree      Test  0.726667   0.759563  0.785311  0.772222  0.792430  0.539790  Good Fit




In [14]:
from sklearn.preprocessing import StandardScaler
feauter_numerik = ['Usia','Tekanan_Darah_Rest','Kolesterol','Detak_Jantung_Max','Oldpeak_ST','BMI']
data_5_pasien = {
    'Usia': [63, 45, 56, 38, 50],
    'Jenis_Kelamin': [3, 1, 0, 1, 0],
    'Tipe_Nyeri_Dada': [-0.3, 1.20, 0.08, -1.05, -0.48],
    'Tekanan_Darah_Rest': [140, 130, 122, 110, 120],
    'Kolesterol': [69, 100, 122, 115, 190],
    'Gula_Darah_Puasa': [1, 0, 0, 0, 1],
    'EKG_Rest': [1, 2, 1, 1, 1],
    'Detak_Jantung_Max': [-0.98, 0.90, -1.05, 0.29, -1.58],
    'Angina_Olahraga': [44, 60, 56, 75, 14],
    'Oldpeak_ST': [19, 59, 73, 76, 18],
    'Kemiringan_ST': [1.28, 2.0, 1.5, 1.5, 2.0],
    'Jumlah_Pembuluh_Darah': [46, 60, 4, 0, 3],
    'Thalassemia': [81, 90, 1, 4, 1],
    'BMI': [10.77, 81.24, 24.5, 28.1, 29.3]
}
scaler = StandardScaler()
data_pasien= pd.DataFrame(data_5_pasien)
data_pasien[feauter_numerik] = scaler.fit_transform(data_pasien[feauter_numerik])
target_asli_pasien = [1, 0, 1, 0, 1] 

data_pasien_baru_x = data_pasien[fitur_terpilih]
data_pasien_baru_y = target_asli_pasien
data_pasien_baru_x

,Jenis_Kelamin,Tipe_Nyeri_Dada,Detak_Jantung_Max,Angina_Olahraga,Oldpeak_ST
0,3,-0.30,-0.535966,44,-1.173811
1,1,1.20,1.495518,60,0.391270
2,0,0.08,-0.611606,56,0.939049
3,1,-1.05,0.836366,75,1.056430
4,0,-0.48,-1.184312,14,-1.212938


In [15]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
# data_pasien_baru = df_x.sample(2, random_state=10)
data_pasien_baru = data_pasien[fitur_terpilih]
prediksi_pasien = best_model_dct.predict(data_pasien_baru)
probabilitas_pasien = best_model_dct.predict_proba(data_pasien_baru)

hasil_df = data_pasien_baru.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model_dct.score(data_pasien_baru_x, data_pasien_baru_y)
prediksi_pasien = best_model_dct.predict(data_pasien_baru_x)
probabilitas_pasien = best_model_dct.predict_proba(data_pasien_baru_x)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))

hasil_df[['Peluang Aman(%)','Peluang Terkena (%)','Prediksi Serangan Jantung']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 80.00%

Tabel Skor Evaluasi Lengkap Data Baru:
        Model Evaluated  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Log Loss
Decision Tree Data_Baru       0.8       0.75     1.0  0.857143      1.0  0.547828


,Peluang Aman(%),Peluang Terkena (%),Prediksi Serangan Jantung
0,10.144928,89.855072,1
1,62.992126,37.007874,0
2,10.144928,89.855072,1
3,14.141414,85.858586,1
4,10.144928,89.855072,1


# 2.Multiclass Classification Decision Trees 